# Persistent Memory & Personalization Demo

This notebook demonstrates the persistent memory system that stores customer interaction history across sessions and provides personalized responses.

In [ ]:
from langchain_core.messages import HumanMessage
from agentic.workflow import orchestrator
from agentic.tools.memory_manager import get_memory_manager
import time

## Part 1: First-Time Customer Interaction

Let's simulate a new customer with their first interaction.

In [ ]:
# First interaction - Login issue
customer_id = "customer_alice@example.com"

test_state = {
    "messages": [HumanMessage(content="I can't log in to my account. I forgot my password.")],
    "customer_id": customer_id
}

config = {
    "configurable": {
        "thread_id": "alice-interaction-1",
    }
}

print("🔵 First Interaction - Login Issue")
print("=" * 60)
result1 = orchestrator.invoke(test_state, config)

# Find the resolver response
for msg in reversed(result1["messages"]):
    if hasattr(msg, 'name') and msg.name == 'resolver':
        print(f"Response: {msg.content[:300]}...")
        break

## Part 2: Second Interaction - Same Customer

The customer returns with a different issue. The system should recognize them and provide personalized service.

In [ ]:
# Wait a moment to simulate time passing
time.sleep(2)

# Second interaction - Subscription question
test_state2 = {
    "messages": [HumanMessage(content="How do I upgrade my subscription to the premium tier?")],
    "customer_id": customer_id
}

config2 = {
    "configurable": {
        "thread_id": "alice-interaction-2",
    }
}

print("\n🟢 Second Interaction - Subscription Question")
print("=" * 60)
result2 = orchestrator.invoke(test_state2, config2)

# Check for returning customer context
for msg in result2["messages"]:
    if hasattr(msg, 'name') and msg.name == 'memory_system':
        print(f"✨ Memory System: {msg.content}")
        print()

# Find the resolver response
for msg in reversed(result2["messages"]):
    if hasattr(msg, 'name') and msg.name == 'resolver':
        print(f"Response: {msg.content[:300]}...")
        break

## Part 3: View Customer History

Let's examine what the system has stored about this customer.

In [ ]:
memory_mgr = get_memory_manager()

# Get customer history
history = memory_mgr.get_customer_history(customer_id, limit=5, include_messages=True)

print("📚 Customer Interaction History")
print("=" * 60)
print(f"Customer: {customer_id}")
print(f"Total interactions: {len(history)}\n")

for i, ticket in enumerate(history, 1):
    print(f"\n{i}. Ticket: {ticket['ticket_id']}")
    print(f"   Category: {ticket['category']}")
    print(f"   Status: {ticket['status']}")
    print(f"   Priority: {ticket['priority']}")
    print(f"   Created: {ticket['created_at']}")
    print(f"   Messages: {ticket.get('message_count', 0)}")
    
    if ticket.get('messages'):
        print("   Conversation:")
        for msg in ticket['messages'][:3]:  # Show first 3 messages
            sender = msg['sender_type']
            content = msg['content'][:100]
            print(f"     [{sender}]: {content}...")

## Part 4: Customer Preferences & Patterns

The system analyzes interaction patterns to understand customer preferences.

In [ ]:
# Get customer preferences
preferences = memory_mgr.get_customer_preferences(customer_id)

print("🎯 Customer Preferences & Patterns")
print("=" * 60)
print(f"Returning Customer: {preferences['is_returning_customer']}")
print(f"Total Interactions: {preferences['total_interactions']}")
print(f"Resolved Tickets: {preferences['resolved_tickets']}")
print(f"Most Common Category: {preferences.get('most_common_category', 'N/A')}")
print(f"Last Interaction: {preferences.get('last_interaction', 'N/A')}")
print(f"\nCategory Distribution: {preferences.get('category_distribution', {})}")
print(f"Priority Distribution: {preferences.get('priority_distribution', {})}")

## Part 5: Find Similar Resolved Issues

The system can find similar past issues to help resolve new ones.

In [ ]:
# Search for similar resolved issues
query = "password reset login problem"
similar_issues = memory_mgr.find_similar_resolved_issues(query, limit=3)

print("🔍 Similar Resolved Issues")
print("=" * 60)
print(f"Query: '{query}'\n")

if similar_issues:
    for i, issue in enumerate(similar_issues, 1):
        print(f"{i}. Ticket: {issue['ticket_id']}")
        print(f"   Category: {issue['category']}")
        print(f"   Subject: {issue['subject']}")
        print(f"   Relevance Score: {issue['relevance_score']}")
        if issue.get('resolution'):
            print(f"   Resolution: {issue['resolution'][:150]}...")
        print()
else:
    print("No similar resolved issues found.")

## Part 6: Third Interaction - Demonstrating Personalization

Let's have another interaction to see how the system uses accumulated history.

In [ ]:
# Third interaction - Another question
test_state3 = {
    "messages": [HumanMessage(content="I want to cancel my reservation for tomorrow.")],
    "customer_id": customer_id
}

config3 = {
    "configurable": {
        "thread_id": "alice-interaction-3",
    }
}

print("\n🟣 Third Interaction - Cancellation Request")
print("=" * 60)
result3 = orchestrator.invoke(test_state3, config3)

# Check for returning customer context
for msg in result3["messages"]:
    if hasattr(msg, 'name') and msg.name == 'memory_system':
        print(f"✨ Memory System: {msg.content}")
        print()

# Find the response
for msg in reversed(result3["messages"]):
    if hasattr(msg, 'name') and msg.name in ['resolver', 'escalation']:
        print(f"Response: {msg.content[:300]}...")
        break

## Part 7: View All Resolved Issues

Let's see all resolved issues in the system for learning and reference.

In [ ]:
# Get all resolved issues
resolved = memory_mgr.get_resolved_issues(limit=10)

print("✅ Resolved Issues Database")
print("=" * 60)
print(f"Total resolved issues: {len(resolved)}\n")

for i, issue in enumerate(resolved[:5], 1):  # Show first 5
    print(f"{i}. Ticket: {issue['ticket_id']}")
    print(f"   Customer: {issue['customer_id']}")
    print(f"   Category: {issue['category']}")
    print(f"   Subject: {issue['subject']}")
    print(f"   Resolved: {issue['resolved_at']}")
    print()

## Summary

This demo shows:

1. **Persistent Storage**: All interactions are saved to the database
2. **Customer Recognition**: Returning customers are identified automatically
3. **Personalization**: Responses consider customer history
4. **Pattern Analysis**: System tracks preferences and common issues
5. **Similar Issue Retrieval**: Past resolutions help solve new problems
6. **Cross-Session Memory**: History persists across different sessions

The memory system enables:
- Better customer experience through personalization
- Faster resolution by learning from past interactions
- Insights into customer behavior and preferences
- Knowledge accumulation over time